In [33]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [53]:
result

{'messages': [HumanMessage(content="Send email to sarniha12@gmail.com with subject 'hi' and body 'you are the best'", additional_kwargs={}, response_metadata={}, id='49a9199b-617c-418d-9f63-234261bc51d2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email via function send_email_tool. Provide required fields.', 'tool_calls': [{'id': 'fc_f70167d1-f818-4bee-96f2-240787a962d2', 'function': {'arguments': '{"body":"you are the best","recipient":"sarniha12@gmail.com","subject":"hi"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 178, 'total_tokens': 244, 'completion_time': 0.136894057, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.00722816, 'prompt_tokens_details': None, 'queue_time': 0.282940357, 'total_time': 0.144122217}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'to

In [57]:
from langgraph.types import Command


if "__interrupt__" in result:
    print("||Paused! Approving...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )
    print(f"Result:{result['messages'][-1].content}")

||Paused! Approving...


BadRequestError: Error code: 400 - {'error': {'message': "'messages' : minimum number of items is 1", 'type': 'invalid_request_error'}}

In [56]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"    

In [55]:

from langchain.chat_models import init_chat_model
agent=create_agent(
    model=init_chat_model("groq:openai/gpt-oss-120b"),


    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [54]:
config={"configurable":{"thread_id":"new-thread-1"}}
result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to sarniha12@gmail.com with subject 'hi' and body 'you are the best'")]},
    config=config
)

In [14]:
cities=["patna","noida","bombay","bangalore","new york"]
for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens,{len(response['messages'])} messages")
    print(f"{(response['messages'])}")

patna: ~167 tokens,8 messages
[HumanMessage(content='Find hotels in patna', additional_kwargs={}, response_metadata={}, id='c9183e12-e114-439e-bdb0-61da5bff6fe5'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'c9nsnb37q', 'function': {'arguments': '{"city":"patna"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 227, 'total_tokens': 243, 'completion_time': 0.03452424, 'completion_tokens_details': None, 'prompt_time': 0.011711711, 'prompt_tokens_details': None, 'queue_time': 0.162536499, 'total_time': 0.046235951}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc16e-c147-7221-ad1d-512858b03132-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'patna'}, 'id': 'c9nsnb37q', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

In [ ]:
###TOKEN SIZENameError                                 Traceback (most recent call last)

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import  InMemorySaver

@tool
def search_hotels(city:str)->str:
    """Search hotels -returns long responses to use more tokens"""
    return f"""Hotels in {city}:
    1.Grand Hotel -5 dtar,$350/night,spa,pool,gym
    2. City Inn- 4 star,$180/night,business center
    3. GAREEB -1 star $23/night,free wifi"""


agent=create_agent(
     model="groq:llama-3.3-70b-versatile",

    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",

            trigger=("tokens",550),
            keep=("tokens",200),
        )
    ]
)
config={"configurable":{"thread_id":"test-1"}}
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//4

NameError: name 'init_chat_model' is not defined

In [8]:
questions=[
    "whats is 4+4?",
    "how to get better at dsa?",
    "whats is 029019*3?",
    "whats the best?",
    "whats is 9*8?",
    "whats is llama?",
]
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to seek assistance with various inquiries, including mathematical calculations and improving skills in Data Structures and Algorithms (DSA).\n\n## SUMMARY\nThe conversation history includes mathematical calculations, such as 4+4 and 029019*3, with results of 8 and 87057, respectively. The user also inquired about improving DSA skills, and a comprehensive response was provided, outlining a 10-step plan that includes building a strong foundation, practicing consistently, using online resources, focusing on common problem patterns, analyzing and learning from mistakes, participating in coding challenges, reading books and articles, joining online communities, working on projects, and taking online courses. Additional tips were provided to supplement the learning process.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user's next steps should involve imp

In [5]:
config={"configurable":{"thread_id":"test-1"}}

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [3]:
###Summarization middleware 

from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent

from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


###MessageBasedSummarization
agent=create_agent(
        model="groq:llama-3.3-70b-versatile",
        checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(
                model="groq:llama-3.3-70b-versatile",
                trigger=("messages",10),
                keep=("messages",4)

            )
        ]

)
 